# ArduMedics Notebook 05: Hyperparameter Sweep & Ablation Study

---

**Notebook:** 05 / 06  |  **Previous:** [NB04 - Model Export & Edge Optimization](./ArduMedics_04_Model_Export_Edge_Optimization.ipynb)  |  **Next:** [NB06 - Full Pipeline Integration & Deployment](./ArduMedics_06_Full_Pipeline_Integration_Deployment.ipynb)

---

## Purpose

This notebook performs **systematic hyperparameter optimization** and **ablation studies** on the best-performing model from Notebooks 01-03. This is **CRITICAL** for the Q1 paper — reviewers want to see that we have thoroughly explored the hyperparameter space and justified every design choice with empirical evidence.

## Ablation & Sweep Coverage

| Study | What We Vary | Why It Matters |
|-------|-------------|----------------|
| Learning Rate Sweep | LR ∈ {0.01, 0.005, 0.001, 0.0005, 0.0001} | Find optimal convergence rate |
| Optimizer Comparison | AdamW vs SGD vs Adam | Choose best gradient descent strategy |
| Augmentation Ablation | No aug / Mosaic / Mosaic+MixUp / Full | Quantify augmentation contribution |
| TPC Window Ablation | Static / W=10 / W=30 / W=50 | Justify Temporal Pose Clustering window |
| OCR Engine Ablation | Individual / Pairwise / Full fusion | Validate multi-engine OCR design |

## Datasets (Same as NB01-03)

### Roboflow Pose Datasets
1. **Falling Pose Estimation** — [https://universe.roboflow.com/humna-pose-data/falling-pose-estimation](https://universe.roboflow.com/humna-pose-data/falling-pose-estimation)
2. **YOLOv8-Pose Fall Detection** — [https://universe.roboflow.com/yolo-xvnzo/yolov8-pose-utovc](https://universe.roboflow.com/yolo-xvnzo/yolov8-pose-utovc)

> **NOTE**: Dataset `nafzzan/falling-pose-estimation-0xme8` was removed as it is a **verified duplicate** of
> Dataset 1 (same 635 images, pixel-identical). Using both would cause data leakage.

### Kaggle Datasets
4. **UR Fall Detection** — [https://www.kaggle.com/datasets/shahliza27/ur-fall-detection-dataset](https://www.kaggle.com/datasets/shahliza27/ur-fall-detection-dataset)
5. **Fall Detection Images** — [https://www.kaggle.com/datasets/uttejkumarkandagatla/fall-detection-dataset](https://www.kaggle.com/datasets/uttejkumarkandagatla/fall-detection-dataset)
6. **Le2i Fall Dataset** — [https://www.kaggle.com/datasets/tuyenldvn/falldataset-imvia](https://www.kaggle.com/datasets/tuyenldvn/falldataset-imvia)
7. **Multiple Cameras Fall** — [https://www.kaggle.com/datasets/soumicksarker/multiple-cameras-fall-dataset](https://www.kaggle.com/datasets/soumicksarker/multiple-cameras-fall-dataset)
8. **Fall Video Dataset** — [https://www.kaggle.com/datasets/payutch/fall-video-dataset](https://www.kaggle.com/datasets/payutch/fall-video-dataset)

## Estimated Runtime

~8-10 hours on NVIDIA T4 GPU (multiple short training runs).

---

## Step 1 of 8: Environment Setup

Install dependencies, import libraries, set random seeds, and load metrics from previous notebooks to determine the best model size.

In [1]:
# ============================================================
# OUTPUT MANAGEMENT UTILITIES (ArduMedics Standard)
# Suppresses noisy output while keeping important logs
# ============================================================

import os, sys, warnings, contextlib, io

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

os.environ['OPENCV_LOG_LEVEL'] = 'ERROR'
os.environ['OPENCV_VIDEOIO_DEBUG'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['FLAGS_logtostderr'] = '0'
os.environ['GLOG_minloglevel'] = '3'
os.environ['YOLO_AUTODOWNLOAD'] = '1'

class suppress_output:
    def __enter__(self):
        self._orig = sys.stdout
        sys.stdout = io.StringIO()
        return self
    def __exit__(self, *a):
        sys.stdout = self._orig
        return False

_real_stdout = sys.stdout
def important_print(msg, end='\n'):
    _real_stdout.write(str(msg) + end)
    _real_stdout.flush()

print('[ArduMedics] Output management loaded')


[ArduMedics] Output management loaded


In [2]:
# ============================================================
# STEP 1a: Install packages + import libraries
# ============================================================

# FIX: Import torch BEFORE pip installs — torch is pre-installed on Kaggle.
# If pip install changes torch version, the already-imported module still works.
# Importing after pip can cause NameError if the reinstall breaks the package.
import torch

!pip install -qq ultralytics roboflow editdistance easyocr pytesseract
# NOTE: Removed opencv-python-headless (conflicts with Kaggle pre-installed cv2)
# NOTE: Removed sahi (not used in this notebook, wastes ~200MB disk)
# NOTE: Removed langchain-* (not used in this notebook, wastes ~300MB disk)
!apt-get install -qq tesseract-ocr

import os, sys, json, yaml, random, shutil, time, glob, warnings, contextlib, io
import cv2
cv2.setLogLevel(0)
import numpy as np

# FIX: to_native() — recursively convert numpy types to Python native types
# for JSON/YAML serialization. Without this, pandas/numpy values like
# np.float64 or np.str_ cause RepresenterError in PyYAML (used by YOLO)
# and TypeError in json.dump().
def to_native(obj):
    """Recursively convert numpy types to Python native types for JSON/YAML serialization."""
    if isinstance(obj, dict):
        return {k: to_native(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [to_native(v) for v in obj]
    elif isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, (np.bool_,)):
        return bool(obj)
    elif isinstance(obj, np.ndarray):
        return to_native(obj.tolist())
    elif isinstance(obj, (np.str_,)):
        return str(obj)
    else:
        return obj

import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from ultralytics import YOLO
from IPython.display import display

warnings.filterwarnings('ignore')

# Random seeds for reproducibility
SEED = 42

# Auto-detect GPU configuration for DDP
# FIX: Also handle CPU-only case (torch.cuda available but no GPU)
if torch.cuda.is_available():
    DEVICE_LIST = [0, 1] if torch.cuda.device_count() >= 2 else [0]
else:
    DEVICE_LIST = 'cpu'
print(f"Using device(s): {DEVICE_LIST}")

# FIX: Set ALL random seeds including torch for full reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# FIX: Batch size must scale with GPU count.
# Original batch=32 was for dual-GPU DDP (16 per GPU).
# On single T4, batch=32 for YOLOv8s-pose can OOM.
BATCH_PER_GPU = 16
SWEEP_BATCH = BATCH_PER_GPU * (len(DEVICE_LIST) if isinstance(DEVICE_LIST, list) else 1)
print(f'Batch size: {SWEEP_BATCH} ({BATCH_PER_GPU}/GPU × {len(DEVICE_LIST) if isinstance(DEVICE_LIST, list) else 1} GPU(s))')

# Base directories
BASE_DIR = Path('/kaggle/working')
DATA_DIR = BASE_DIR / 'unified_fall_dataset'
RESULTS_DIR = BASE_DIR / 'ablation_results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Ultralytics: {__import__("ultralytics").__version__}')
print(f'PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
print(f'SEED: {SEED}')
print(f'DATA_DIR: {DATA_DIR}')
print(f'RESULTS_DIR: {RESULTS_DIR}')
print('✓ Step 1a complete: Environment ready')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 123.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo s

In [3]:
# ============================================================
# STEP 1b: Load metrics from previous notebooks + determine best model
# ============================================================
# 
# CRITICAL: On Kaggle, /kaggle/working/ is wiped between sessions.
# Previous notebook outputs must be mounted as Kaggle Datasets.
# We search BOTH /kaggle/working/ (same session) AND /kaggle/input/ (mounted datasets).
#
# Expected Kaggle Dataset names to add as input:
#   - ardumedics-nb01-nano-outputs   (contains best_nano.pt + metrics JSON)
#   - ardumedics-nb02-small-outputs  (contains best_small.pt + metrics JSON)  
#   - ardumedics-nb03-medium-outputs (contains best_medium.pt + metrics JSON)
#   - ardumedics-nb04-ocr-outputs    (contains OCR metrics JSON)

# BASE_DIR, RESULTS_DIR, DATA_DIR, SEED are defined in Step 1a above

# ── Search for metrics JSON files in ALL possible locations ──
search_dirs = [
    BASE_DIR,                    # Same session output
    Path('/kaggle/input'),       # Mounted Kaggle datasets
]

metrics_files = []
for search_dir in search_dirs:
    if search_dir.exists():
        # Direct files
        metrics_files.extend(search_dir.glob('metrics_notebook0*.json'))
        # Nested in subdirectories (Kaggle dataset mount structure)
        metrics_files.extend(search_dir.rglob('*/metrics_notebook0*.json'))
        # Deeper nesting
        metrics_files.extend(search_dir.rglob('*/*/metrics_notebook0*.json'))

# Deduplicate
metrics_files = sorted(set(metrics_files))
print(f"Found {len(metrics_files)} metrics files:")
for mf in metrics_files:
    print(f"  {mf}")

# ── Extract best model size from metrics ──
best_model_size = 'n'  # default: nano (safest for ablation sweeps under 12 hours)
best_map50 = 0.0
model_map50 = {}  # size → mAP50

for mf in metrics_files:
    try:
        with open(mf, 'r') as f:
            m = json.load(f)
        
        # Extract mAP50 - handle ALL possible JSON formats:
        # Format 1: NB01/NB03 nested: m['val']['box_map50']
        # Format 2: NB02 (old) flat: m['val_mAP50']  
        # Format 3: Standardized: m['best_mAP50']
        map50 = 0.0
        if 'best_mAP50' in m:
            map50 = m['best_mAP50']
        elif 'val' in m and isinstance(m['val'], dict):
            map50 = m['val'].get('box_map50', 0)
        elif 'val_mAP50' in m:
            map50 = m['val_mAP50']
        
        # Extract model size - handle ALL possible formats:
        # Format 1: m['model_size'] = 'n'/'s'/'m' (standardized)
        # Format 2: m['model'] = 'yolov8n-pose' → extract first letter after 'v8'
        size = 'n'
        if 'model_size' in m:
            size = m['model_size']
        elif 'model' in m:
            model_name = m['model']
            if 'nano' in model_name or model_name.endswith('n-pose'):
                size = 'n'
            elif 'small' in model_name or model_name.endswith('s-pose'):
                size = 's'
            elif 'medium' in model_name or model_name.endswith('m-pose'):
                size = 'm'
        
        # Skip non-pose metrics (e.g., NB04 OCR has no mAP50 for pose)
        if map50 == 0.0 and 'best_mAP50' not in m and 'val' not in m:
            print(f"  {mf.name}: Skipping (no pose metrics)")
            continue
        
        print(f"  {mf.name}: model_size={size}, mAP50={map50:.4f}")
        
        if map50 > best_map50:
            best_map50 = map50
            best_model_size = size
        
        if size not in model_map50 or map50 > model_map50.get(size, 0):
            model_map50[size] = map50
            
    except Exception as e:
        print(f"  Could not load {mf}: {e}")

print(f"\nModel mAP50 summary: {model_map50}")
print(f">>> Best model size from previous notebooks: {best_model_size} (mAP50={best_map50:.4f})")

# ── TIME SAFETY: Cap model size for 12-hour Kaggle limit ──
# Medium model with 180 sweep epochs can exceed 12 hours on T4.
# Only use nano or small for ablation sweeps.
if best_model_size == 'm':
    if 's' in model_map50 and model_map50['s'] > 0:
        best_model_size = 's'
        print(f"  ⚠ Capping model size from 'm' to 's' for 12h time limit (mAP50={model_map50['s']:.4f})")
    elif 'n' in model_map50 and model_map50['n'] > 0:
        best_model_size = 'n'
        print(f"  ⚠ Capping model size from 'm' to 'n' for 12h time limit (mAP50={model_map50['n']:.4f})")
    else:
        best_model_size = 'n'
        print(f"  ⚠ Defaulting to 'n' for 12h time limit")

print(f">>> Using yolov8{best_model_size}-pose for ablation sweeps")

# ── Locate trained model weights from NB01/02/03 ──
# These are needed for TPC evaluation (Step 6)
trained_models = {}
model_filenames = {
    'n': ['best_nano.pt', 'best.pt'],
    's': ['best_small.pt', 'best.pt'],
    'm': ['best_medium.pt', 'best.pt'],
}

for search_dir in search_dirs:
    if search_dir.exists():
        for size, filenames in model_filenames.items():
            if size in trained_models:
                continue
            for fname in filenames:
                # Search for the model file
                candidates = list(search_dir.rglob(f'*/{fname}')) + list(search_dir.rglob(f'{fname}'))
                for cand in candidates:
                    # Check if this is the right model for this size
                    path_str = str(cand).lower()
                    if size == 'n' and ('nano' in path_str or 'nb01' in path_str):
                        trained_models[size] = str(cand)
                        break
                    elif size == 's' and ('small' in path_str or 'nb02' in path_str):
                        trained_models[size] = str(cand)
                        break
                    elif size == 'm' and ('medium' in path_str or 'nb03' in path_str):
                        trained_models[size] = str(cand)
                        break
                if size in trained_models:
                    break

# Also check the runs directory from same session
for size, run_name in [('n', 'ardumedics_nano_pose'), ('s', 'ardumedics_small_pose'), ('m', 'ardumedics_medium_pose')]:
    if size not in trained_models:
        run_path = f'/kaggle/working/runs/{run_name}/weights/best.pt'
        if os.path.exists(run_path):
            trained_models[size] = run_path

print(f"\nTrained models found:")
for size, path in trained_models.items():
    print(f"  {size}: {path}")

# Select the best trained model for TPC evaluation
best_trained_model = None
if best_model_size in trained_models:
    best_trained_model = trained_models[best_model_size]
    print(f"\n>>> Best trained model for TPC: {best_trained_model}")
else:
    print(f"\n>>> WARNING: No trained model found for size '{best_model_size}'")
    print("    TPC evaluation will use pretrained weights (less accurate)")


Found 8 metrics files:
  /kaggle/input/notebooks/nishatfifa/ardumedics-01-yolov8n-pose-fall-detection-training/metrics_notebook01_nano.json
  /kaggle/input/notebooks/nishatfifa/ardumedics-01-yolov8n-pose-fall-detection-training/nb01_outputs/metrics_notebook01_nano.json
  /kaggle/input/notebooks/nishatfifa/ardumedics-02-yolov8s-pose-fall-detection-training/metrics_notebook02_small.json
  /kaggle/input/notebooks/nishatfifa/ardumedics-02-yolov8s-pose-fall-detection-training/nb02_outputs/metrics_notebook02_small.json
  /kaggle/input/notebooks/nishatfifa/ardumedics-03-yolov8m-pose-fall-detection-training/metrics_notebook03_medium.json
  /kaggle/input/notebooks/nishatfifa/ardumedics-03-yolov8m-pose-fall-detection-training/nb03_outputs/metrics_notebook03_medium.json
  /kaggle/input/notebooks/nishatfifa/ardumedics-04-ocr-multiengine-prescription-analysi/nb04_outputs/metrics_notebook04_ocr.json
  /kaggle/input/notebooks/nishatfifa/ardumedics-04-ocr-multiengine-prescription-analysi/ocr_results/m

## Step 2 of 8: Recreate Unified Dataset

Download and merge all 7 datasets (Nafzzan dataset removed — verified duplicate) (identical pipeline to NB01-03). If the unified dataset already exists from a previous notebook, skip download.

In [4]:
# Notebook: 05 | Step: 2 of 8
# Recreate Unified Dataset — Download, Merge, Create data.yaml
# After this: Learning Rate Sweep (Step 3)

# Step 2: Recreate Unified Dataset
# ROBUST: Recursively scan ALL of /kaggle/input/ for data.yaml
# Then COPY to /kaggle/working/ (writable!) — NEVER use /kaggle/input/ as working dir

# FIX: Refactored merge logic into a function so it can be called BOTH after
# finding local datasets AND after downloading via Roboflow SDK.
# Previously, if datasets were downloaded, the merge never ran, leaving
# DATA_DIR empty and all subsequent training would fail with "No images found".

def remap_label_file(lbl_file, class_map, dest_dir, dest_name=None):
    """Remap class IDs in a YOLO label file."""
    if dest_name is None:
        dest_name = Path(lbl_file).name
    dest_file = dest_dir / dest_name
    if not class_map:
        shutil.copy2(lbl_file, dest_file)
        return
    with open(lbl_file, 'r') as f:
        lines = f.readlines()
    remapped = []
    for line in lines:
        parts = line.strip().split()
        if not parts:
            continue
        old_cls = int(parts[0])
        if old_cls in class_map:
            parts[0] = str(class_map[old_cls])
            remapped.append(' '.join(parts))
    with open(dest_file, 'w') as f:
        f.write('\n'.join(remapped))

def merge_roboflow_datasets():
    """Merge roboflow_0/ and roboflow_1/ into unified DATA_DIR."""
    # Create unified dataset directories
    for split in ['train', 'val', 'test']:
        (DATA_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
        (DATA_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)
    
    # Collect and merge
    pairs = []
    roboflow_configs = [
        {'path': BASE_DIR / 'roboflow_0', 'class_map': {0: 0}},  # person → fall
        {'path': BASE_DIR / 'roboflow_1', 'class_map': {}},      # already unified
    ]
    for cfg in roboflow_configs:
        ds_dir = cfg['path']
        if not ds_dir.exists():
            continue
        for split in ['train', 'valid', 'val', 'test']:
            img_dir = ds_dir / split / 'images'
            lbl_dir = ds_dir / split / 'labels'
            if not img_dir.exists():
                continue
            for img_file in img_dir.glob('*'):
                if img_file.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
                    lbl_file = lbl_dir / (img_file.stem + '.txt')
                    if lbl_file.exists() and lbl_file.stat().st_size > 0:
                        pairs.append((img_file, lbl_file, cfg['class_map']))
    
    if not pairs:
        print('  WARNING: No image-label pairs found during merge!')
        return
    
    # Shuffle and split
    random.seed(SEED)
    random.shuffle(pairs)
    val_count = int(len(pairs) * 0.15)
    test_count = int(len(pairs) * 0.05)
    val_pairs = pairs[:val_count]
    test_pairs = pairs[val_count:val_count+test_count]
    train_pairs = pairs[val_count+test_count:]
    
    for pair_list, split_name in [(train_pairs, 'train'), (val_pairs, 'val'), (test_pairs, 'test')]:
        for img_file, lbl_file, class_map in pair_list:
            unique_name = f"{img_file.parent.stem}_{img_file.name}"
            shutil.copy2(img_file, DATA_DIR / 'images' / split_name / unique_name)
            remap_label_file(lbl_file, class_map, DATA_DIR / 'labels' / split_name,
                           dest_name=unique_name.replace(img_file.suffix, '.txt'))
    
    print(f"Unified dataset: {len(train_pairs)} train, {len(val_pairs)} val, {len(test_pairs)} test")

_need_download = False

# Check for existing unified dataset
if DATA_DIR.exists() and (DATA_DIR / 'data.yaml').exists():
    print(f"Unified dataset already exists at {DATA_DIR}. Skipping.")
else:
    DATASET1_PATTERNS = ['falling_pose_estimation', 'falling-pose-estimation',
                         'Falling pose estimation', 'humna']
    DATASET2_PATTERNS = ['yolov8_pose_fall', 'yolov8-pose',
                         'yolov8-pose.v1i', 'yolov8-pose.v1xme8', 'yolo-xvnzo']
    
    found_ds1 = None
    found_ds2 = None
    
    if Path('/kaggle/input').exists():
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'data.yaml' in files:
                root_lower = root.lower()
                if any(p.lower() in root_lower for p in DATASET1_PATTERNS):
                    found_ds1 = root
                    print(f"  Found Dataset 1 at: {root}")
                elif any(p.lower() in root_lower for p in DATASET2_PATTERNS):
                    found_ds2 = root
                    print(f"  Found Dataset 2 at: {root}")
                else:
                    if found_ds1 is None:
                        found_ds1 = root
                    elif found_ds2 is None:
                        found_ds2 = root
    
    # Copy found datasets to writable directory
    if found_ds1:
        dst = BASE_DIR / 'roboflow_0'
        if not dst.exists():
            shutil.copytree(found_ds1, str(dst))
        print(f"  Dataset 1 → roboflow_0/")
    
    if found_ds2:
        dst = BASE_DIR / 'roboflow_1'
        if not dst.exists():
            shutil.copytree(found_ds2, str(dst))
        print(f"  Dataset 2 → roboflow_1/")
    
    if not found_ds1 and not found_ds2:
        _need_download = True
    else:
        # FIX: Call merge function when datasets found locally
        merge_roboflow_datasets()

# FIX: After downloading, ALSO merge the datasets into unified format
if _need_download:
    print("[Fallback] Attempting Roboflow SDK download...")
    api_key = os.environ.get('ROBOFLOW_API_KEY', '')
    if not api_key:
        raise ValueError("No datasets found locally and ROBOFLOW_API_KEY not set. "
                         "Upload datasets or set the environment variable.")
    try:
        from roboflow import Roboflow
        rf = Roboflow(api_key=api_key)
        p1 = rf.workspace("humna-pose-data").project("falling-pose-estimation")
        ds1 = p1.version(2).download("yolov8", location=str(BASE_DIR / 'roboflow_0'))
        p2 = rf.workspace("yolo-xvnzo").project("yolov8-pose-utovc")
        ds2 = p2.version(3).download("yolov8", location=str(BASE_DIR / 'roboflow_1'))
        # FIX: Now merge the downloaded datasets
        merge_roboflow_datasets()
    except Exception as e:
        print(f"Roboflow SDK download failed: {e}")
        raise ValueError("No datasets available. Upload data or set ROBOFLOW_API_KEY env var.")

# ─── Ensure data_yaml_path and IMG_DIR are defined for downstream cells ──────
IMG_DIR = DATA_DIR / 'images'
data_yaml_path = DATA_DIR / 'data.yaml'

# ─── Create/update data.yaml with kpt_shape and flip_idx (REQUIRED for pose) ─
data_yaml = {
    'path': str(DATA_DIR),
    'train': str(IMG_DIR / 'train'),
    'val': str(IMG_DIR / 'val'),
    'test': str(IMG_DIR / 'test'),
    'nc': 2,
    'names': {0: 'fall', 1: 'not-fallen'},
    'kpt_shape': [17, 3],
    'flip_idx': [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15],
}
DATA_DIR.mkdir(parents=True, exist_ok=True)
with open(data_yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f'\ndata.yaml written to {data_yaml_path}')
print(f'  kpt_shape: {data_yaml["kpt_shape"]}')
print(f'  flip_idx:  {data_yaml["flip_idx"]}')
for s in ['train', 'val', 'test']:
    img_dir = IMG_DIR / s
    if img_dir.exists():
        n_imgs = len(list(img_dir.glob('*.*')))
        print(f'  {s}: {n_imgs} images')

print('\n>>> Dataset ready for hyperparameter sweeps.')

  Found Dataset 1 at: /kaggle/input/datasets/nishatfifa/ardumedics-roboflow-pose-datasets/ardumedics-roboflow-pose-datasets/falling_pose_estimation
  Found Dataset 2 at: /kaggle/input/datasets/nishatfifa/ardumedics-roboflow-pose-datasets/ardumedics-roboflow-pose-datasets/yolov8_pose_fall
  Dataset 1 → roboflow_0/
  Dataset 2 → roboflow_1/
Unified dataset: 819 train, 153 val, 51 test

data.yaml written to /kaggle/working/unified_fall_dataset/data.yaml
  kpt_shape: [17, 3]
  flip_idx:  [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]
  train: 819 images
  val: 153 images
  test: 51 images

>>> Dataset ready for hyperparameter sweeps.


## Step 3 of 8: Learning Rate Sweep

Train the best model size with 5 different learning rates using shorter runs (15 epochs each) with early stopping. Record final mAP@50 for each LR and plot the LR vs mAP curve.

In [5]:
# Notebook: 05 | Step: 3 of 8
# Learning Rate Sweep — Train with LR ∈ {0.01, 0.005, 0.001, 0.0005, 0.0001}
# After this: Optimizer Comparison (Step 4)

LEARNING_RATES = [0.01, 0.005, 0.001, 0.0005, 0.0001]
SWEEP_EPOCHS = 15
# FIX: SWEEP_BATCH is now computed in Step 1a based on GPU count (16/GPU)
# Was hardcoded to 32 which OOMs on single-GPU T4 with YOLOv8s-pose

lr_results = []

for lr in LEARNING_RATES:
    print(f'\n{"="*60}')
    print(f'  LEARNING RATE SWEEP: lr = {lr}')
    print(f'{"="*60}')

    # Load pretrained model (fresh weights each time)
    model = YOLO(f'yolov8{best_model_size}-pose.pt')

    # Train with current LR
    # FIX: Replace dots in run_name (0.01 → 0_01) to avoid directory issues
    run_name = f'lr_sweep_lr{str(lr).replace(".", "_")}'
    results = model.train(
        data=str(data_yaml_path),
        epochs=SWEEP_EPOCHS,
        imgsz=640,
        batch=SWEEP_BATCH,
        lr0=lr,
        lrf=lr / 10,          # final LR = lr0/10
        patience=8,           # early stopping patience
        project=str(RESULTS_DIR / 'lr_sweep'),
        name=run_name,
        seed=SEED,
        verbose=False,
        device=DEVICE_LIST,
        workers=4,
        exist_ok=True,
    )

    # Extract metrics
    metrics = model.val(data=str(data_yaml_path), verbose=False)
    map50 = metrics.box.map50
    map50_95 = metrics.box.map

    lr_results.append({
        'learning_rate': lr,
        'mAP50': float(round(map50, 4)),
        'mAP50-95': float(round(map50_95, 4)),
    })
    print(f'  LR={lr} | mAP50={map50:.4f} | mAP50-95={map50_95:.4f}')

# ─── Summary DataFrame ──────────────────────────────────────────────
lr_df = pd.DataFrame(lr_results)
display(lr_df)

# ─── Plot: LR vs mAP50 Curve ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(lr_df['learning_rate'], lr_df['mAP50'], 'o-', color='#2196F3', linewidth=2, markersize=8)
ax.set_xscale('log')
ax.set_xlabel('Learning Rate (log scale)', fontsize=12)
ax.set_ylabel('mAP@50', fontsize=12)
ax.set_title('Learning Rate vs mAP@50 — ArduMedics Fall Detection', fontsize=14)
ax.grid(True, alpha=0.3)
best_lr_row = lr_df.loc[lr_df['mAP50'].idxmax()]
ax.axvline(best_lr_row['learning_rate'], color='red', linestyle='--', alpha=0.5,
           label=f'Best LR={best_lr_row["learning_rate"]}')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'lr_vs_map50.png', dpi=200)
plt.show()

BEST_LR = float(best_lr_row['learning_rate'])
print(f'\n>>> Best learning rate: {BEST_LR} (mAP50={best_lr_row["mAP50"]})')

# Save LR sweep results
with open(RESULTS_DIR / 'lr_sweep_results.json', 'w') as f:
    json.dump(to_native(lr_results), f, indent=2)


  LEARNING RATE SWEEP: lr = 0.01
Ultralytics 8.4.53 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/unified_fall_dataset/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-pose.pt, momentum=0.937

,learning_rate,mAP50,mAP50-95
0,0.0100,0.7746,0.5141
1,0.0050,0.7716,0.5224
2,0.0010,0.7575,0.4822
3,0.0005,0.6086,0.3421
4,0.0001,0.7761,0.4998



>>> Best learning rate: 0.0001 (mAP50=0.7761)


## Step 4 of 8: Optimizer Comparison

Compare AdamW, SGD, and Adam optimizers (3 training runs). Same short training of 15 epochs each. Record mAP, training time, and convergence speed. Plot training curves side by side.

In [6]:
# Notebook: 05 | Step: 4 of 8
# Optimizer Comparison — AdamW vs SGD vs Adam
# After this: Augmentation Ablation (Step 5)

OPTIMIZERS = ['AdamW', 'SGD', 'Adam']
OPT_EPOCHS = 15
OPT_BATCH = SWEEP_BATCH  # FIX: Use auto-scaled batch from Step 1a (was hardcoded 32)

opt_results = []
opt_curves = {}  # store per-epoch metrics for plotting

for optimizer in OPTIMIZERS:
    print(f'\n{"="*60}')
    print(f'  OPTIMIZER COMPARISON: {optimizer}')
    print(f'{"="*60}')

    # Fresh pretrained model each time
    model = YOLO(f'yolov8{best_model_size}-pose.pt')

    run_name = f'opt_sweep_{optimizer.lower()}'
    t0 = time.time()

    results = model.train(
        data=str(data_yaml_path),
        epochs=OPT_EPOCHS,
        imgsz=640,
        batch=OPT_BATCH,
        lr0=BEST_LR,
        optimizer=optimizer,
        patience=8,
        project=str(RESULTS_DIR / 'opt_sweep'),
        name=run_name,
        seed=SEED,
        verbose=False,
        device=DEVICE_LIST,
        workers=4,
        exist_ok=True,
    )
    train_time = time.time() - t0

    # Validation metrics
    metrics = model.val(data=str(data_yaml_path), verbose=False)
    map50 = metrics.box.map50
    map50_95 = metrics.box.map

    # Load training log for convergence curves
    csv_path = RESULTS_DIR / 'opt_sweep' / run_name / 'results.csv'
    epochs_to_converge = OPT_EPOCHS  # default
    if csv_path.exists():
        try:
            log_df = pd.read_csv(csv_path)
            log_df.columns = [c.strip() for c in log_df.columns]
            opt_curves[optimizer] = log_df
            # Find epoch where validation loss was lowest (convergence)
            val_loss_col = [c for c in log_df.columns if 'val/box_loss' in c]
            if val_loss_col:
                best_epoch = log_df[val_loss_col[0]].idxmin() + 1
                epochs_to_converge = int(best_epoch)
        except Exception as e:
            print(f'  Could not parse training log: {e}')

    opt_results.append({
        'optimizer': optimizer,
        'mAP50': float(round(map50, 4)),
        'mAP50-95': float(round(map50_95, 4)),
        'training_time_min': round(train_time / 60, 1),
        'epochs_to_converge': epochs_to_converge,
    })
    print(f'  {optimizer}: mAP50={map50:.4f} | Time={train_time/60:.1f}min | Converged@epoch={epochs_to_converge}')

# ─── Summary DataFrame ──────────────────────────────────────────────
opt_df = pd.DataFrame(opt_results)
display(opt_df)

# ─── Plot: Training Curves Side by Side ─────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metric_cols = {
    'train/box_loss': 'Training Box Loss',
    'val/box_loss': 'Validation Box Loss',
    'metrics/mAP50(B)': 'mAP@50',
}
colors = {'AdamW': '#4CAF50', 'SGD': '#2196F3', 'Adam': '#FF9800'}

for ax, (col, title) in zip(axes, metric_cols.items()):
    for opt_name, log_df in opt_curves.items():
        matching = [c for c in log_df.columns if col in c]
        if matching:
            ax.plot(log_df.index + 1, log_df[matching[0]],
                    label=opt_name, color=colors.get(opt_name, 'gray'), linewidth=2)
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel(title, fontsize=11)
    ax.set_title(title, fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('Optimizer Comparison — Training Curves (ArduMedics)', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'optimizer_training_curves.png', dpi=200, bbox_inches='tight')
plt.show()

# Determine best optimizer
best_opt_row = opt_df.loc[opt_df['mAP50'].idxmax()]
BEST_OPTIMIZER = str(best_opt_row['optimizer'])
print(f'\n>>> Best optimizer: {BEST_OPTIMIZER} (mAP50={best_opt_row["mAP50"]})')

# Save optimizer results
with open(RESULTS_DIR / 'optimizer_results.json', 'w') as f:
    json.dump(to_native(opt_results), f, indent=2)


  OPTIMIZER COMPARISON: AdamW
Ultralytics 8.4.53 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/unified_fall_dataset/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-pose.pt, momentum=0.937, 

,optimizer,mAP50,mAP50-95,training_time_min,epochs_to_converge
0,AdamW,0.7382,0.5311,3.2,14
1,SGD,0.7366,0.5132,3.1,14
2,Adam,0.7366,0.5307,3.1,13



>>> Best optimizer: AdamW (mAP50=0.7382)


## Step 5 of 8: Augmentation Ablation

Run 4 augmentation configurations to quantify each augmentation's contribution:
- **(a)** No augmentation (baseline)
- **(b)** Mosaic only
- **(c)** Mosaic + MixUp
- **(d)** Full augmentation (Mosaic + MixUp + Copy-Paste)

Each configuration runs for 15 epochs.

In [7]:
# Notebook: 05 | Step: 5 of 8
# Augmentation Ablation — No aug / Mosaic / Mosaic+MixUp / Full aug
# After this: TPC Ablation Study (Step 6)

AUG_EPOCHS = 15
AUG_BATCH = SWEEP_BATCH  # FIX: Use auto-scaled batch from Step 1a (was hardcoded 32)

aug_configs = [
    {
        'name': 'no_augmentation',
        'label': 'No Augmentation',
        'mosaic': 0.0,
        'mixup': 0.0,
        'copy_paste': 0.0,
        'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.0,
        'degrees': 0.0, 'translate': 0.0, 'scale': 0.0, 'flipud': 0.0, 'fliplr': 0.0,
    },
    {
        'name': 'mosaic_only',
        'label': 'Mosaic Only',
        'mosaic': 1.0,
        'mixup': 0.0,
        'copy_paste': 0.0,
    },
    {
        'name': 'mosaic_mixup',
        'label': 'Mosaic + MixUp',
        'mosaic': 1.0,
        'mixup': 0.1,
        'copy_paste': 0.0,
    },
    {
        'name': 'full_augmentation',
        'label': 'Full Aug (Mosaic+MixUp+CopyPaste)',
        'mosaic': 1.0,
        'mixup': 0.1,
        'copy_paste': 0.1,
    },
]

aug_results = []

for aug_cfg in aug_configs:
    print(f'\n{"="*60}')
    print(f'  AUGMENTATION ABLATION: {aug_cfg["label"]}')
    print(f'{"="*60}')

    model = YOLO(f'yolov8{best_model_size}-pose.pt')

    run_name = f'aug_ablation_{aug_cfg["name"]}'

    # Build train kwargs with augmentation params
    train_kwargs = dict(
        data=str(data_yaml_path),
        epochs=AUG_EPOCHS,
        imgsz=640,
        batch=AUG_BATCH,
        lr0=BEST_LR,
        optimizer=BEST_OPTIMIZER,
        patience=8,
        project=str(RESULTS_DIR / 'aug_ablation'),
        name=run_name,
        seed=SEED,
        verbose=False,
        device=DEVICE_LIST,
        workers=4,
        exist_ok=True,
        close_mosaic=5,  # disable mosaic for last 5 epochs
    )

    # Add augmentation-specific params
    for key in ['mosaic', 'mixup', 'copy_paste', 'hsv_h', 'hsv_s', 'hsv_v',
                'degrees', 'translate', 'scale', 'flipud', 'fliplr']:
        if key in aug_cfg:
            train_kwargs[key] = aug_cfg[key]

    results = model.train(**train_kwargs)

    # Validation
    metrics = model.val(data=str(data_yaml_path), verbose=False)
    map50 = metrics.box.map50
    map50_95 = metrics.box.map

    aug_results.append({
        'config': aug_cfg['name'],
        'label': aug_cfg['label'],
        'mosaic': aug_cfg.get('mosaic', 0),
        'mixup': aug_cfg.get('mixup', 0),
        'copy_paste': aug_cfg.get('copy_paste', 0),
        'mAP50': float(round(map50, 4)),
        'mAP50-95': float(round(map50_95, 4)),
    })
    print(f'  {aug_cfg["label"]}: mAP50={map50:.4f} | mAP50-95={map50_95:.4f}')

# ─── Summary DataFrame ──────────────────────────────────────────────
aug_df = pd.DataFrame(aug_results)
display(aug_df)

# ─── Plot: Augmentation Ablation Bar Chart ──────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(aug_results))
width = 0.35
bars1 = ax.bar(x - width/2, [r['mAP50'] for r in aug_results], width,
               label='mAP@50', color='#2196F3', alpha=0.85)
bars2 = ax.bar(x + width/2, [r['mAP50-95'] for r in aug_results], width,
               label='mAP@50-95', color='#FF9800', alpha=0.85)

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Augmentation Ablation — ArduMedics Fall Detection', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels([r['label'] for r in aug_results], rotation=15, ha='right')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'augmentation_ablation.png', dpi=200)
plt.show()

print(f'\n>>> Augmentation ablation complete.')

# Save augmentation results
with open(RESULTS_DIR / 'augmentation_results.json', 'w') as f:
    json.dump(to_native(aug_results), f, indent=2)


  AUGMENTATION ABLATION: No Augmentation
Ultralytics 8.4.53 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/unified_fall_dataset/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-pose.pt, momentum

,config,label,mosaic,mixup,copy_paste,mAP50,mAP50-95
0,no_augmentation,No Augmentation,0.0,0.0,0.0,0.7007,0.4917
1,mosaic_only,Mosaic Only,1.0,0.0,0.0,0.7254,0.5094
2,mosaic_mixup,Mosaic + MixUp,1.0,0.1,0.0,0.7723,0.5379
3,full_augmentation,Full Aug (Mosaic+MixUp+CopyPaste),1.0,0.1,0.1,0.7723,0.5379



>>> Augmentation ablation complete.


## Step 6 of 8: TPC Ablation Study (Fall Detection Logic)

**This is KEY for the paper** — proving our Temporal Pose Clustering (TPC) contribution works.

We compare:
- **(a)** Static pose detection only (no temporal context)
- **(b)** TPC with window=10
- **(c)** TPC with window=30 (default)
- **(d)** TPC with window=50

Metrics: Fall detection accuracy, False Positive Rate (FPR), False Negative Rate (FNR).

In [8]:
# Notebook: 05 | Step: 6 of 8
# TPC Ablation Study — Temporal Pose Clustering window size comparison
# After this: OCR Engine Ablation (Step 7)

from collections import deque
# torch already imported in Step 1a

# ─── Load Best Trained Model ────────────────────────────────────────
# Try to find best model from sweeps, else use default
# CRITICAL: Use the FULLY-TRAINED model from NB02/NB03, NOT a 15-epoch sweep model.
# The ablation sweep models are only trained for 15 epochs and would give terrible TPC results.
if best_trained_model and os.path.exists(best_trained_model):
    best_model_path = best_trained_model
    print(f'Loading fully-trained model from previous notebook: {best_model_path}')
else:
    # Last resort: search for any trained model in /kaggle/input/
    model_candidates = []
    for search_dir in [Path('/kaggle/input'), Path('/kaggle/working')]:
        if search_dir.exists():
            model_candidates.extend(search_dir.rglob('*/best*.pt'))
            model_candidates.extend(search_dir.rglob('best*.pt'))
    if model_candidates:
        best_model_path = str(model_candidates[0])
        print(f'Found trained model: {best_model_path}')
    else:
        best_model_path = f'yolov8{best_model_size}-pose.pt'
        print(f'WARNING: No trained model found. Using pretrained: {best_model_path}')

model = YOLO(best_model_path)

# ─── TPC Implementation ─────────────────────────────────────────────
class TemporalPoseCluster:
    """
    Temporal Pose Clustering (TPC) module for fall detection.
    Aggregates pose keypoints over a sliding window to reduce
    false positives from single-frame misclassifications.
    """
    def __init__(self, window_size=30, fall_threshold=0.5, confidence_threshold=0.5):
        self.window_size = window_size
        self.fall_threshold = fall_threshold
        self.confidence_threshold = confidence_threshold
        self.pose_buffer = deque(maxlen=window_size)
        self.fall_scores = deque(maxlen=window_size)

    def _compute_fall_score(self, keypoints):
        """
        Compute a fall score from keypoints.
        Based on: head-hip vertical distance, torso angle, aspect ratio.
        
        keypoints: numpy array of shape (17, 3) — [x, y, confidence] per keypoint
        """
        if keypoints is None or len(keypoints) < 17:
            return 0.0

        kp = keypoints[:17]  # COCO 17 keypoints

        # Extract key points (COCO format) — now includes confidence
        nose = kp[0][:2]
        left_shoulder = kp[5][:2]
        right_shoulder = kp[6][:2]
        left_hip = kp[11][:2]
        right_hip = kp[12][:2]

        # FIX: Visibility check now works — keypoints have (x, y, conf) format
        # Previously used keypoints.xy (17×2) so len(k)>2 was always False,
        # making the visibility check permanently bypassed.
        vis = [k[2] if len(k) > 2 else 1.0 for k in kp[:13]]
        if sum(vis[:13]) / 13 < 0.3:
            return 0.0

        # 1. Head-Hip vertical distance (normalized by image height)
        head_y = nose[1]
        hip_y = (left_hip[1] + right_hip[1]) / 2
        head_hip_dist = abs(head_y - hip_y)
        # In standing: head is above hip → small head_hip_dist in normalized coords
        # In falling: head is closer to hip level → smaller vertical distance
        fall_score_height = 1.0 - min(head_hip_dist / 0.5, 1.0)  # normalize

        # 2. Torso angle (shoulder midpoint to hip midpoint)
        shoulder_mid = ((left_shoulder[0] + right_shoulder[0]) / 2,
                        (left_shoulder[1] + right_shoulder[1]) / 2)
        hip_mid = ((left_hip[0] + right_hip[0]) / 2,
                   (left_hip[1] + right_hip[1]) / 2)
        dx = hip_mid[0] - shoulder_mid[0]
        dy = hip_mid[1] - shoulder_mid[1]
        torso_angle = abs(np.arctan2(dx, dy))  # 0 = vertical (standing), pi/2 = horizontal
        fall_score_angle = torso_angle / (np.pi / 2)  # 0=standing, 1=horizontal

        # Combined fall score
        fall_score = 0.5 * fall_score_height + 0.5 * fall_score_angle
        return min(max(fall_score, 0.0), 1.0)

    def update(self, keypoints_list):
        """
        Update TPC with new frame's keypoints.
        Returns: (is_fall, confidence, temporal_score)
        """
        if keypoints_list is None or len(keypoints_list) == 0:
            self.pose_buffer.append(None)
            self.fall_scores.append(0.0)
            return False, 0.0, 0.0

        # Process the first detected person (primary subject)
        kpts = keypoints_list[0]
        fall_score = self._compute_fall_score(kpts)

        self.pose_buffer.append(kpts)
        self.fall_scores.append(fall_score)

        # Temporal aggregation: average fall score over window
        valid_scores = [s for s in self.fall_scores if s > 0]
        if len(valid_scores) < max(1, self.window_size // 3):
            return False, fall_score, 0.0

        temporal_score = np.mean(valid_scores)
        is_fall = temporal_score > self.fall_threshold
        confidence = temporal_score

        return is_fall, confidence, temporal_score


def run_tpc_evaluation(model, data_yaml_path, window_size, num_samples=200):
    """
    Run TPC evaluation on test frames with given window size.
    window_size=0 means static (no temporal context).
    """
    # Load test images
    with open(data_yaml_path) as f:
        data_cfg = yaml.safe_load(f)
    test_dir = Path(data_cfg.get('test', data_cfg.get('val', '')))
    if not test_dir.exists():
        test_dir = Path(data_cfg['val'])

    test_images = list(test_dir.glob('*.*'))[:num_samples]
    if len(test_images) == 0:
        return {'accuracy': 0, 'fpr': 0, 'fnr': 0, 'num_samples': 0}

    # Initialize TPC (window_size=1 effectively disables temporal)
    if window_size == 0:
        tpc = TemporalPoseCluster(window_size=1, fall_threshold=0.5)
    else:
        tpc = TemporalPoseCluster(window_size=window_size, fall_threshold=0.5)

    tp, fp, tn, fn = 0, 0, 0, 0

    for img_path in test_images:
        # Ground truth: check label file
        label_path = Path(str(img_path).replace('images', 'labels')).with_suffix('.txt')
        gt_fall = False
        if label_path.exists():
            with open(label_path) as lf:
                for line in lf:
                    parts = line.strip().split()
                    if len(parts) > 0 and int(parts[0]) == 0:  # class 0 = fall
                        gt_fall = True
                        break

        # Run inference
        results = model.predict(str(img_path), verbose=False, conf=0.25)

        # FIX: Extract BOTH keypoints.xy AND keypoints.conf to build (x, y, conf) arrays.
        # Previously only keypoints.xy was extracted (shape 17×2), so the visibility
        # check in _compute_fall_score was always bypassed (len(k)=2, never >2).
        if len(results) > 0 and results[0].keypoints is not None:
            kpts_xy = results[0].keypoints.xy.cpu().numpy()    # (N, 17, 2)
            kpts_conf = results[0].keypoints.conf.cpu().numpy() # (N, 17)
            if len(kpts_xy) > 0:
                # Combine xy + conf → shape (N, 17, 3) so visibility check works
                kpts_with_conf = []
                for i in range(len(kpts_xy)):
                    combined = np.concatenate([
                        kpts_xy[i],                         # (17, 2)
                        kpts_conf[i][:, np.newaxis]         # (17, 1)
                    ], axis=1)                                # (17, 3)
                    kpts_with_conf.append(combined)
                is_fall, conf, temporal_score = tpc.update(kpts_with_conf)
            else:
                is_fall = False
        else:
            is_fall = False

        # Update confusion matrix
        if gt_fall and is_fall:
            tp += 1
        elif not gt_fall and is_fall:
            fp += 1
        elif not gt_fall and not is_fall:
            tn += 1
        elif gt_fall and not is_fall:
            fn += 1

    # Compute metrics
    total = tp + fp + tn + fn
    accuracy = (tp + tn) / max(total, 1)
    fpr = fp / max(fp + tn, 1)
    fnr = fn / max(fn + tp, 1)

    return {
        'accuracy': round(accuracy, 4),
        'fpr': round(fpr, 4),
        'fnr': round(fnr, 4),
        'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn,
        'num_samples': total,
    }


# ─── Run TPC Ablation ───────────────────────────────────────────────
tpc_configs = [
    {'name': 'Static (no TPC)', 'window_size': 0, 'label': 'Static (no temporal)'},
    {'name': 'TPC W=10', 'window_size': 10, 'label': 'TPC W=10'},
    {'name': 'TPC W=30 (default)', 'window_size': 30, 'label': 'TPC W=30 (default)'},
    {'name': 'TPC W=50', 'window_size': 50, 'label': 'TPC W=50'},
]

tpc_results = []

for tpc_cfg in tpc_configs:
    print(f'\n{"="*60}')
    print(f'  TPC ABLATION: {tpc_cfg["label"]}')
    print(f'{"="*60}')

    metrics = run_tpc_evaluation(
        model, str(data_yaml_path), tpc_cfg['window_size'], num_samples=200
    )
    metrics['config'] = tpc_cfg['name']
    metrics['label'] = tpc_cfg['label']
    metrics['window_size'] = tpc_cfg['window_size']
    tpc_results.append(metrics)

    print(f'  Accuracy={metrics["accuracy"]:.4f} | FPR={metrics["fpr"]:.4f} | FNR={metrics["fnr"]:.4f}')
    print(f'  TP={metrics["tp"]} | FP={metrics["fp"]} | TN={metrics["tn"]} | FN={metrics["fn"]}')

# ─── Summary DataFrame ──────────────────────────────────────────────
tpc_df = pd.DataFrame(tpc_results)
display(tpc_df[['label', 'window_size', 'accuracy', 'fpr', 'fnr', 'num_samples']])

# ─── Plot: TPC Window Size vs Accuracy / FPR ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
ax = axes[0]
ax.plot([r['window_size'] for r in tpc_results],
        [r['accuracy'] for r in tpc_results],
        'o-', color='#4CAF50', linewidth=2, markersize=10)
ax.set_xlabel('TPC Window Size (0 = Static)', fontsize=12)
ax.set_ylabel('Fall Detection Accuracy', fontsize=12)
ax.set_title('TPC Window Size vs Accuracy', fontsize=13)
ax.set_xticks([r['window_size'] for r in tpc_results])
ax.set_xticklabels([r['label'] for r in tpc_results], rotation=15)
ax.grid(True, alpha=0.3)

# FPR / FNR plot
ax = axes[1]
x = np.arange(len(tpc_results))
width = 0.35
ax.bar(x - width/2, [r['fpr'] for r in tpc_results], width,
       label='FPR (False Positive Rate)', color='#F44336', alpha=0.8)
ax.bar(x + width/2, [r['fnr'] for r in tpc_results], width,
       label='FNR (False Negative Rate)', color='#FF9800', alpha=0.8)
ax.set_ylabel('Rate', fontsize=12)
ax.set_title('TPC Window Size vs FPR / FNR', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels([r['label'] for r in tpc_results], rotation=15)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.suptitle('TPC Ablation Study — ArduMedics Fall Detection', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'tpc_ablation.png', dpi=200, bbox_inches='tight')
plt.show()

print(f'\n>>> TPC ablation study complete.')

# Save TPC results
with open(RESULTS_DIR / 'tpc_ablation_results.json', 'w') as f:
    json.dump(to_native(tpc_results), f, indent=2)

Loading fully-trained model from previous notebook: /kaggle/input/notebooks/nishatfifa/ardumedics-02-yolov8s-pose-fall-detection-training/nb02_outputs/best_small.pt

  TPC ABLATION: Static (no temporal)
  Accuracy=0.7451 | FPR=0.2549 | FNR=0.0000
  TP=0 | FP=13 | TN=38 | FN=0

  TPC ABLATION: TPC W=10
  Accuracy=0.9020 | FPR=0.0980 | FNR=0.0000
  TP=0 | FP=5 | TN=46 | FN=0

  TPC ABLATION: TPC W=30 (default)
  Accuracy=1.0000 | FPR=0.0000 | FNR=0.0000
  TP=0 | FP=0 | TN=51 | FN=0

  TPC ABLATION: TPC W=50
  Accuracy=1.0000 | FPR=0.0000 | FNR=0.0000
  TP=0 | FP=0 | TN=51 | FN=0


,label,window_size,accuracy,fpr,fnr,num_samples
0,Static (no temporal),0,0.7451,0.2549,0.0,51
1,TPC W=10,10,0.9020,0.0980,0.0,51
2,TPC W=30 (default),30,1.0000,0.0000,0.0,51
3,TPC W=50,50,1.0000,0.0000,0.0,51



>>> TPC ablation study complete.


## Step 7 of 8: OCR Engine Ablation

Compare individual OCR engines vs pairwise combinations vs full fusion.
Measure Character Error Rate (CER) and Word Error Rate (WER) for each combination.
This validates our multi-engine OCR fusion design for the medication label reader.

In [9]:
# Notebook: 05 | Step: 7 of 8
# OCR Engine Ablation — Individual / Pairwise / Full Fusion
# After this: Generate All Ablation Tables and Figures (Step 8)

import editdistance

# ─── OCR Engine Wrappers ────────────────────────────────────────────
class OCREngineBase:
    """Base class for OCR engines."""
    def __init__(self, name):
        self.name = name
    def recognize(self, image):
        raise NotImplementedError

class TesseractOCR(OCREngineBase):
    def __init__(self):
        super().__init__('Tesseract')
        import pytesseract
        self.pytesseract = pytesseract
    def recognize(self, image):
        try:
            text = self.pytesseract.image_to_string(image, config='--psm 6')
            return text.strip()
        except Exception:
            return ''

class EasyOCREngine(OCREngineBase):
    def __init__(self):
        super().__init__('EasyOCR')
        import easyocr
        self.reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())
    def recognize(self, image):
        try:
            results = self.reader.readtext(image)
            text = ' '.join([r[1] for r in results])
            return text.strip()
        except Exception:
            return ''

class YOLOv8OCR(OCREngineBase):
    """
    Placeholder for YOLOv8-based text detection + recognition.
    In production, this would use a YOLOv8 text detection model.
    For ablation, we simulate with a simple OpenCV approach.
    """
    def __init__(self):
        super().__init__('YOLOv8-OCR')
    def recognize(self, image):
        try:
            # Simple OpenCV-based text extraction as proxy
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image
            _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            # Use Tesseract on preprocessed image as proxy
            import pytesseract
            text = pytesseract.image_to_string(binary, config='--psm 6')
            return text.strip()
        except Exception:
            return ''


def compute_cer(predicted, ground_truth):
    """Character Error Rate."""
    if len(ground_truth) == 0:
        return 0.0 if len(predicted) == 0 else 1.0
    return editdistance.eval(predicted, ground_truth) / len(ground_truth)

def compute_wer(predicted, ground_truth):
    """Word Error Rate."""
    pred_words = predicted.split()
    gt_words = ground_truth.split()
    if len(gt_words) == 0:
        return 0.0 if len(pred_words) == 0 else 1.0
    return editdistance.eval(pred_words, gt_words) / len(gt_words)


def ocr_fusion(texts):
    """
    Simple voting-based fusion of multiple OCR outputs.
    For each word position, take the most common output across engines.
    """
    if len(texts) == 0:
        return ''
    if len(texts) == 1:
        return texts[0]

    # Simple approach: use longest common subsequence-like merge
    # For ablation purposes, we use majority voting at word level
    all_words = [t.split() for t in texts if t]
    if not all_words:
        return ''

    max_len = max(len(w) for w in all_words)
    fused = []
    for i in range(max_len):
        candidates = []
        for word_list in all_words:
            if i < len(word_list):
                candidates.append(word_list[i])
        if candidates:
            # Most common word at this position
            from collections import Counter
            most_common = Counter(candidates).most_common(1)[0][0]
            fused.append(most_common)

    return ' '.join(fused)


# ─── Create Synthetic Medication Label Test Set ─────────────────────
# Generate simple test images with known text for CER/WER evaluation
test_texts = [
    'ASPIRIN 500MG',
    'AMOXICILLIN 250MG',
    'IBUPROFEN 400MG',
    'PARACETAMOL 500MG',
    'OMEPRAZOLE 20MG',
    'METFORMIN 500MG',
    'LISINOPRIL 10MG',
    'ATORVASTATIN 20MG',
    'TAKE ONE TABLET DAILY',
    'TAKE TWICE DAILY WITH FOOD',
]

def create_text_image(text, size=(400, 100)):
    """Create a simple image with text for OCR testing."""
    img = np.ones((*size[::-1], 3), dtype=np.uint8) * 255
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.8
    thickness = 2
    text_size = cv2.getTextSize(text, font, font_scale, thickness)[0]
    x = (size[0] - text_size[0]) // 2
    y = (size[1] + text_size[1]) // 2
    cv2.putText(img, text, (x, y), font, font_scale, (0, 0, 0), thickness)
    return img

test_images = [create_text_image(t) for t in test_texts]
print(f'Created {len(test_images)} synthetic medication label images for OCR testing.')

# ─── Initialize OCR Engines ─────────────────────────────────────────
print('Initializing OCR engines...')
engines = {}
try:
    engines['Tesseract'] = TesseractOCR()
    print('  Tesseract: OK')
except Exception as e:
    print(f'  Tesseract: FAILED - {e}')

try:
    engines['EasyOCR'] = EasyOCREngine()
    print('  EasyOCR: OK')
except Exception as e:
    print(f'  EasyOCR: FAILED - {e}')

try:
    engines['YOLOv8-OCR'] = YOLOv8OCR()
    print('  YOLOv8-OCR: OK')
except Exception as e:
    print(f'  YOLOv8-OCR: FAILED - {e}')

print(f'\nInitialized {len(engines)} OCR engines: {list(engines.keys())}')

# ─── Define Ablation Configurations ─────────────────────────────────
engine_names = list(engines.keys())
ocr_configs = []

# Individual engines
for name in engine_names:
    ocr_configs.append({'label': name, 'engines': [name]})

# Pairwise combinations
from itertools import combinations
for combo in combinations(engine_names, 2):
    ocr_configs.append({'label': ' + '.join(combo), 'engines': list(combo)})

# Full fusion
ocr_configs.append({'label': ' + '.join(engine_names) + ' (Full)', 'engines': engine_names})

print(f'\nOCR ablation configurations: {len(ocr_configs)}')
for cfg in ocr_configs:
    print(f'  - {cfg["label"]}')

# ─── Run OCR Ablation ───────────────────────────────────────────────
ocr_results = []

for cfg in ocr_configs:
    print(f'\n  Evaluating: {cfg["label"]}')
    cer_list, wer_list = [], []

    for img, gt_text in zip(test_images, test_texts):
        # Run each engine in this configuration
        texts = []
        for eng_name in cfg['engines']:
            if eng_name in engines:
                try:
                    t = engines[eng_name].recognize(img)
                    texts.append(t)
                except Exception:
                    texts.append('')

        # Apply fusion if multiple engines
        if len(texts) > 1:
            pred_text = ocr_fusion(texts)
        elif len(texts) == 1:
            pred_text = texts[0]
        else:
            pred_text = ''

        cer_list.append(compute_cer(pred_text.upper(), gt_text.upper()))
        wer_list.append(compute_wer(pred_text.upper(), gt_text.upper()))

    avg_cer = np.mean(cer_list)
    avg_wer = np.mean(wer_list)

    ocr_results.append({
        'config': cfg['label'],
        'num_engines': len(cfg['engines']),
        'CER': round(avg_cer, 4),
        'WER': round(avg_wer, 4),
    })
    print(f'    CER={avg_cer:.4f} | WER={avg_wer:.4f}')

# ─── Summary DataFrame ──────────────────────────────────────────────
ocr_df = pd.DataFrame(ocr_results)
display(ocr_df)

print(f'\n>>> OCR engine ablation complete.')

# Save OCR results
with open(RESULTS_DIR / 'ocr_ablation_results.json', 'w') as f:
    json.dump(to_native(ocr_results), f, indent=2)

Created 10 synthetic medication label images for OCR testing.
Initializing OCR engines...
  Tesseract: OK
  EasyOCR: OK
  YOLOv8-OCR: OK

Initialized 3 OCR engines: ['Tesseract', 'EasyOCR', 'YOLOv8-OCR']

OCR ablation configurations: 7
  - Tesseract
  - EasyOCR
  - YOLOv8-OCR
  - Tesseract + EasyOCR
  - Tesseract + YOLOv8-OCR
  - EasyOCR + YOLOv8-OCR
  - Tesseract + EasyOCR + YOLOv8-OCR (Full)

  Evaluating: Tesseract
    CER=0.0607 | WER=0.1500

  Evaluating: EasyOCR
    CER=0.0789 | WER=0.3500

  Evaluating: YOLOv8-OCR
    CER=0.0607 | WER=0.1500

  Evaluating: Tesseract + EasyOCR
    CER=0.0607 | WER=0.1500

  Evaluating: Tesseract + YOLOv8-OCR
    CER=0.0607 | WER=0.1500

  Evaluating: EasyOCR + YOLOv8-OCR
    CER=0.0789 | WER=0.3500

  Evaluating: Tesseract + EasyOCR + YOLOv8-OCR (Full)
    CER=0.0607 | WER=0.1500


,config,num_engines,CER,WER
0,Tesseract,1,0.0607,0.15
1,EasyOCR,1,0.0789,0.35
2,YOLOv8-OCR,1,0.0607,0.15
3,Tesseract + EasyOCR,2,0.0607,0.15
4,Tesseract + YOLOv8-OCR,2,0.0607,0.15
5,EasyOCR + YOLOv8-OCR,2,0.0789,0.35
6,Tesseract + EasyOCR + YOLOv8-OCR (Full),3,0.0607,0.15



>>> OCR engine ablation complete.


## Step 8 of 8: Generate All Ablation Tables and Figures

Generate LaTeX tables for the Q1 paper and save all figures to `/kaggle/working/ablation_results/`. Also save `metrics_notebook05_ablation.json`.

In [10]:
# Notebook: 05 | Step: 8 of 8
# Generate All Ablation Tables (LaTeX) and Figures for Paper
# After this: Notebook 05 complete → Proceed to NB06 (Full Pipeline Integration)

# ═══════════════════════════════════════════════════════════════════════
# TABLE 1: Learning Rate Sweep Results
# ═══════════════════════════════════════════════════════════════════════
lr_latex = r"""\begin{table}[htbp]
\centering
\caption{Learning rate sweep results for YOLOv8""" + f"{best_model_size}-pose" + r""" on the unified fall detection dataset.}
\label{tab:lr_sweep}
\begin{tabular}{lc}
\toprule
Learning Rate & mAP@50 \\
\midrule
"""

for r in lr_results:
    lr_latex += f"{r['learning_rate']} & {r['mAP50']:.4f} \\\\\n"

lr_latex += r"""\bottomrule
\end{tabular}
\end{table}
"""

print('Table 1: Learning Rate Sweep')
print(lr_latex)

with open(RESULTS_DIR / 'table_lr_sweep.tex', 'w') as f:
    f.write(lr_latex)

# ═══════════════════════════════════════════════════════════════════════
# TABLE 2: Optimizer Comparison
# ═══════════════════════════════════════════════════════════════════════
opt_latex = r"""\begin{table}[htbp]
\centering
\caption{Optimizer comparison for YOLOv8""" + f"{best_model_size}-pose" + r""" training (30 epochs).}
\label{tab:optimizer_comparison}
\begin{tabular}{lcccc}
\toprule
Optimizer & mAP@50 & mAP@50-95 & Training Time (min) & Convergence Epoch \\
\midrule
"""

for r in opt_results:
    opt_latex += (f"{r['optimizer']} & {r['mAP50']:.4f} & {r['mAP50-95']:.4f} "
                  f"& {r['training_time_min']:.1f} & {r['epochs_to_converge']} \\\\\n")

opt_latex += r"""\bottomrule
\end{tabular}
\end{table}
"""

print('\nTable 2: Optimizer Comparison')
print(opt_latex)

with open(RESULTS_DIR / 'table_optimizer_comparison.tex', 'w') as f:
    f.write(opt_latex)

# ═══════════════════════════════════════════════════════════════════════
# TABLE 3: Augmentation Ablation
# ═══════════════════════════════════════════════════════════════════════
aug_latex = r"""\begin{table}[htbp]
\centering
\caption{Augmentation ablation study results (15 epochs each).}
\label{tab:augmentation_ablation}
\begin{tabular}{lccccc}
\toprule
Configuration & Mosaic & MixUp & Copy-Paste & mAP@50 & mAP@50-95 \\
\midrule
"""

for r in aug_results:
    aug_latex += (f"{r['label']} & {r['mosaic']} & {r['mixup']} & {r['copy_paste']} "
                  f"& {r['mAP50']:.4f} & {r['mAP50-95']:.4f} \\\\\n")

aug_latex += r"""\bottomrule
\end{tabular}
\end{table}
"""

print('\nTable 3: Augmentation Ablation')
print(aug_latex)

with open(RESULTS_DIR / 'table_augmentation_ablation.tex', 'w') as f:
    f.write(aug_latex)

# ═══════════════════════════════════════════════════════════════════════
# TABLE 4: TPC Window Size Ablation
# ═══════════════════════════════════════════════════════════════════════
tpc_latex = r"""\begin{table}[htbp]
\centering
\caption{Temporal Pose Clustering (TPC) window size ablation study.}
\label{tab:tpc_ablation}
\begin{tabular}{lcccc}
\toprule
Configuration & Window Size & Accuracy & FPR & FNR \\
\midrule
"""

for r in tpc_results:
    tpc_latex += (f"{r['label']} & {r['window_size']} & {r['accuracy']:.4f} "
                  f"& {r['fpr']:.4f} & {r['fnr']:.4f} \\\\\n")

tpc_latex += r"""\bottomrule
\end{tabular}
\end{table}
"""

print('\nTable 4: TPC Window Size Ablation')
print(tpc_latex)

with open(RESULTS_DIR / 'table_tpc_ablation.tex', 'w') as f:
    f.write(tpc_latex)

# ═══════════════════════════════════════════════════════════════════════
# TABLE 5: OCR Engine Ablation
# ═══════════════════════════════════════════════════════════════════════
ocr_latex = r"""\begin{table}[htbp]
\centering
\caption{OCR engine ablation study on synthetic medication labels.}
\label{tab:ocr_ablation}
\begin{tabular}{lccc}
\toprule
Configuration & \# Engines & CER & WER \\
\midrule
"""

for r in ocr_results:
    ocr_latex += (f"{r['config']} & {r['num_engines']} & {r['CER']:.4f} & {r['WER']:.4f} \\\\\n")

ocr_latex += r"""\bottomrule
\end{tabular}
\end{table}
"""

print('\nTable 5: OCR Engine Ablation')
print(ocr_latex)

with open(RESULTS_DIR / 'table_ocr_ablation.tex', 'w') as f:
    f.write(ocr_latex)

# ═══════════════════════════════════════════════════════════════════════
# COMPREHENSIVE FIGURE: All Ablation Results Overview
# ═══════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# (a) LR vs mAP50
ax = axes[0, 0]
ax.plot(lr_df['learning_rate'], lr_df['mAP50'], 'o-', color='#2196F3', linewidth=2, markersize=8)
ax.set_xscale('log')
ax.set_xlabel('Learning Rate (log scale)')
ax.set_ylabel('mAP@50')
ax.set_title('(a) Learning Rate Sweep')
ax.grid(True, alpha=0.3)

# (b) Optimizer comparison bar
ax = axes[0, 1]
opt_names = [r['optimizer'] for r in opt_results]
opt_maps = [r['mAP50'] for r in opt_results]
bars = ax.bar(opt_names, opt_maps, color=['#4CAF50', '#2196F3', '#FF9800'])
ax.set_ylabel('mAP@50')
ax.set_title('(b) Optimizer Comparison')
for bar, val in zip(bars, opt_maps):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', fontsize=9)
ax.grid(axis='y', alpha=0.3)

# (c) Augmentation ablation bar
ax = axes[1, 0]
aug_labels = [r['label'] for r in aug_results]
aug_maps = [r['mAP50'] for r in aug_results]
colors_aug = ['#F44336', '#FF9800', '#FFEB3B', '#4CAF50']
bars = ax.barh(aug_labels, aug_maps, color=colors_aug[:len(aug_results)])
ax.set_xlabel('mAP@50')
ax.set_title('(c) Augmentation Ablation')
for bar, val in zip(bars, aug_maps):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)
ax.grid(axis='x', alpha=0.3)

# (d) TPC window vs accuracy
ax = axes[1, 1]
tpc_windows = [str(r['window_size']) for r in tpc_results]
tpc_acc = [r['accuracy'] for r in tpc_results]
tpc_fpr = [r['fpr'] for r in tpc_results]
ax.plot(tpc_windows, tpc_acc, 'o-', color='#4CAF50', linewidth=2, label='Accuracy')
ax.plot(tpc_windows, tpc_fpr, 's--', color='#F44336', linewidth=2, label='FPR')
ax.set_xlabel('TPC Window Size')
ax.set_ylabel('Score')
ax.set_title('(d) TPC Window Size Ablation')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('ArduMedics — Comprehensive Ablation Study Results', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'ablation_overview_composite.png', dpi=200, bbox_inches='tight')
plt.show()

print(f'\n>>> All figures saved to {RESULTS_DIR}')

Table 1: Learning Rate Sweep
\begin{table}[htbp]
\centering
\caption{Learning rate sweep results for YOLOv8s-pose on the unified fall detection dataset.}
\label{tab:lr_sweep}
\begin{tabular}{lc}
\toprule
Learning Rate & mAP@50 \\
\midrule
0.01 & 0.7746 \\
0.005 & 0.7716 \\
0.001 & 0.7575 \\
0.0005 & 0.6086 \\
0.0001 & 0.7761 \\
\bottomrule
\end{tabular}
\end{table}


Table 2: Optimizer Comparison
\begin{table}[htbp]
\centering
\caption{Optimizer comparison for YOLOv8s-pose training (30 epochs).}
\label{tab:optimizer_comparison}
\begin{tabular}{lcccc}
\toprule
Optimizer & mAP@50 & mAP@50-95 & Training Time (min) & Convergence Epoch \\
\midrule
AdamW & 0.7382 & 0.5311 & 3.2 & 14 \\
SGD & 0.7366 & 0.5132 & 3.1 & 14 \\
Adam & 0.7366 & 0.5307 & 3.1 & 13 \\
\bottomrule
\end{tabular}
\end{table}


Table 3: Augmentation Ablation
\begin{table}[htbp]
\centering
\caption{Augmentation ablation study results (15 epochs each).}
\label{tab:augmentation_ablation}
\begin{tabular}{lccccc}
\toprule
Confi

In [11]:
# Notebook: 05 | Step: 8 of 8 (continued)
# Save metrics_notebook05_ablation.json — Final Metrics Summary
# After this: Notebook 05 complete → Proceed to NB06

from datetime import datetime  # already imported in Step 1a, safe to re-import

# ─── Compile All Results ─────────────────────────────────────────────
all_metrics = {
    'notebook': 'ArduMedics_05_Hyperparameter_Sweep_Ablation',
    'notebook_index': '05/06',
    'timestamp': datetime.now().isoformat(),
    'seed': SEED,
    'model_size': best_model_size,
    'best_learning_rate': BEST_LR,
    'best_optimizer': BEST_OPTIMIZER,
    'lr_sweep': lr_results,
    'optimizer_comparison': opt_results,
    'augmentation_ablation': aug_results,
    'tpc_ablation': tpc_results,
    'ocr_ablation': ocr_results,
    'paper_artifacts': {
        'latex_tables': [
            'table_lr_sweep.tex',
            'table_optimizer_comparison.tex',
            'table_augmentation_ablation.tex',
            'table_tpc_ablation.tex',
            'table_ocr_ablation.tex',
        ],
        'figures': [
            'lr_vs_map50.png',
            'optimizer_training_curves.png',
            'augmentation_ablation.png',
            'tpc_ablation.png',
            'ablation_overview_composite.png',
        ],
    },
}

# ─── Save to JSON ───────────────────────────────────────────────────
metrics_path = BASE_DIR / 'metrics_notebook05_ablation.json'
with open(metrics_path, 'w') as f:
    json.dump(to_native(all_metrics), f, indent=2)

print(f'Metrics saved to: {metrics_path}')

# ─── List All Generated Artifacts ───────────────────────────────────
print(f'\n{"="*60}')
print('  NOTEBOOK 05 — ALL GENERATED ARTIFACTS')
print(f'{"="*60}')

for item in sorted(RESULTS_DIR.rglob('*')):
    if item.is_file():
        size_kb = item.stat().st_size / 1024
        print(f'  {item.relative_to(RESULTS_DIR)} ({size_kb:.1f} KB)')

print(f'\n{"="*60}')
print('  NOTEBOOK 05 COMPLETE — SUMMARY')
print(f'{"="*60}')
print(f'  Best learning rate  : {BEST_LR}')
print(f'  Best optimizer      : {BEST_OPTIMIZER}')
print(f'  Best model size     : {best_model_size}')
print(f'  LR sweep runs       : {len(lr_results)}')
print(f'  Optimizer runs      : {len(opt_results)}')
print(f'  Augmentation configs: {len(aug_results)}')
print(f'  TPC window configs  : {len(tpc_results)}')
print(f'  OCR engine configs  : {len(ocr_results)}')
print(f'  LaTeX tables        : 5')
print(f'  Publication figures : 5')
print(f'\n  >>> Next: NB06 - Full Pipeline Integration & Deployment')
print(f'{"="*60}')

Metrics saved to: /kaggle/working/metrics_notebook05_ablation.json

  NOTEBOOK 05 — ALL GENERATED ARTIFACTS
  ablation_overview_composite.png (261.6 KB)
  aug_ablation/aug_ablation_full_augmentation/BoxF1_curve.png (154.0 KB)
  aug_ablation/aug_ablation_full_augmentation/BoxPR_curve.png (102.7 KB)
  aug_ablation/aug_ablation_full_augmentation/BoxP_curve.png (143.9 KB)
  aug_ablation/aug_ablation_full_augmentation/BoxR_curve.png (137.4 KB)
  aug_ablation/aug_ablation_full_augmentation/PoseF1_curve.png (132.1 KB)
  aug_ablation/aug_ablation_full_augmentation/PosePR_curve.png (102.7 KB)
  aug_ablation/aug_ablation_full_augmentation/PoseP_curve.png (151.3 KB)
  aug_ablation/aug_ablation_full_augmentation/PoseR_curve.png (109.4 KB)
  aug_ablation/aug_ablation_full_augmentation/args.yaml (1.7 KB)
  aug_ablation/aug_ablation_full_augmentation/confusion_matrix.png (99.4 KB)
  aug_ablation/aug_ablation_full_augmentation/confusion_matrix_normalized.png (111.4 KB)
  aug_ablation/aug_ablation_full

---
## Step 9: Save Outputs for Next Notebook

Pack all ablation outputs for use by Notebook 06 (Final Results).

**Instructions**: After this notebook completes, create a new Kaggle Dataset
from the output. Name it `ardumedics-nb05-ablation-outputs`.

In [12]:
# ============================================================
# STEP 9: Save outputs for next notebook
# ============================================================

OUTPUT_PACK_DIR = '/kaggle/working/nb05_outputs'
os.makedirs(OUTPUT_PACK_DIR, exist_ok=True)

# Copy metrics JSON (the key file NB06 looks for)
metrics_src = str(BASE_DIR / 'metrics_notebook05_ablation.json')
if os.path.exists(metrics_src):
    shutil.copy2(metrics_src, f'{OUTPUT_PACK_DIR}/metrics_notebook05_ablation.json')
    print('  Copied metrics_notebook05_ablation.json')

# Copy ablation result JSONs
for json_name in ['lr_sweep_results.json', 'optimizer_results.json', 
                  'augmentation_results.json', 'tpc_ablation_results.json', 'ocr_ablation_results.json']:
    src = str(RESULTS_DIR / json_name)
    if os.path.exists(src):
        shutil.copy2(src, f'{OUTPUT_PACK_DIR}/{json_name}')
        print(f'  Copied {json_name}')

# Copy figures
fig_dir = str(RESULTS_DIR)
dst_fig = f'{OUTPUT_PACK_DIR}/figures'
if os.path.exists(fig_dir) and not os.path.exists(dst_fig):
    os.makedirs(dst_fig, exist_ok=True)
    for ext in ['*.png', '*.tex']:
        for f in glob.glob(os.path.join(fig_dir, ext)):
            shutil.copy2(f, dst_fig)
    print(f'  Copied figures/ and tables/')

print(f'\nAll outputs packed to: {OUTPUT_PACK_DIR}')
print('\n>>> IMPORTANT: Save this output as a Kaggle Dataset! <<<')
print('    Name it: ardumedics-nb05-ablation-outputs')
print('    Add this dataset as input to NB06')
print('\n\u2713 Step 9 complete: Outputs packed for next notebook')

  Copied metrics_notebook05_ablation.json
  Copied lr_sweep_results.json
  Copied optimizer_results.json
  Copied augmentation_results.json
  Copied tpc_ablation_results.json
  Copied ocr_ablation_results.json
  Copied figures/ and tables/

All outputs packed to: /kaggle/working/nb05_outputs

>>> IMPORTANT: Save this output as a Kaggle Dataset! <<<
    Name it: ardumedics-nb05-ablation-outputs
    Add this dataset as input to NB06

✓ Step 9 complete: Outputs packed for next notebook
